
# 03 — Foreign Exchange Data Quality Validation

## Project: Financial Market Intelligence & Currency Risk Analytics

### Business Objective

The Treasury and Finance teams depend on reliable historical exchange rates to monitor currency risk and estimate the INR cost of international import payments.

Incorrect, missing, or duplicated exchange rates could distort financial calculations, risk indicators, and management dashboards.

This notebook validates the extracted USD/INR, EUR/INR, and GBP/INR datasets before loading them into PostgreSQL.

## Data Source

- Provider: Frankfurter API
- Input file: `Data/raw/fx_rates_raw.csv`
- Currency pairs: USD/INR, EUR/INR, GBP/INR
- Expected records: 312
- Historical period: 2026-05-21 to 2026-10-13

## Data Quality Checks

1. **Schema validation:** Confirm required columns and data types.
2. **Missing-value validation:** Identify null or missing values.
3. **Duplicate validation:** Detect repeated currency-date combinations.
4. **Exchange-rate validation:** Identify zero, negative, or invalid rates.
5. **Currency validation:** Confirm only approved currency pairs are present.
6. **Date validation:** Check historical date boundaries and date coverage.
7. **Outlier screening:** Identify unusual exchange-rate movements requiring investigation.
8. **Reconciliation:** Confirm record counts and completeness against the saved ingestion metadata.

## Business Rules

- Each currency pair must have no more than one exchange rate per observation date.
- Exchange rates must be positive numerical values.
- Currency pairs must belong to the approved USD/INR, EUR/INR, and GBP/INR list.
- Observation dates must fall within the requested historical period.
- Weekends and market holidays may legitimately have no observations.
- Unusual exchange-rate movements must be flagged for review rather than automatically deleted.


In [1]:

from pathlib import Path
import sys

# Locate the project root.
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parents[1]
elif project_root.name.lower() == "python":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Python.Scripts.pipeline_logger import get_logger

logger = get_logger("FX_Data_Validation")

logger.info("FX data validation notebook initialized")


2026-11-02 16:04:55,459 | INFO | FX_Data_Validation | FX data validation notebook initialized



## 1. Load Raw FX Data and Ingestion Metadata

### Objective
Load the previously extracted foreign exchange dataset and its ingestion metadata for systematic data-quality validation.

### Method
1. Identify the project root directory.
2. Locate the saved raw FX CSV and metadata JSON files.
3. Load the CSV into a Pandas DataFrame.
4. Parse observation dates into datetime format.
5. Load ingestion metadata for reconciliation.
6. Verify that both files contain the expected information.

### Validation
- Confirm both source files exist.
- Confirm the CSV contains the required columns.
- Compare the loaded record count with the ingestion metadata.
- Confirm the expected currency pairs are represented.

In [1]:

from pathlib import Path
import json

import pandas as pd

# Identify project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

# Define source file paths
RAW_DATA_DIR = PROJECT_ROOT / "Data" / "raw"

FX_CSV_PATH = RAW_DATA_DIR / "fx_rates_raw.csv"
FX_METADATA_PATH = RAW_DATA_DIR / "fx_ingestion_metadata.json"

# Verify source files exist
if not FX_CSV_PATH.is_file():
    raise FileNotFoundError(f"Missing FX CSV: {FX_CSV_PATH}")

if not FX_METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing ingestion metadata: {FX_METADATA_PATH}"
    )

# Load raw FX dataset
df_fx_raw = pd.read_csv(
    FX_CSV_PATH,
    parse_dates=["observation_date"]
)

# Load ingestion metadata
with open(FX_METADATA_PATH, "r", encoding="utf-8") as file:
    fx_metadata = json.load(file)

# Define required schema
REQUIRED_COLUMNS = [
    "observation_date",
    "base_currency",
    "quote_currency",
    "exchange_rate",
    "source"
]

# Validate required columns
missing_columns = set(REQUIRED_COLUMNS) - set(df_fx_raw.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

# Reconcile record count with metadata
expected_records = fx_metadata["total_records"]
actual_records = len(df_fx_raw)

assert actual_records == expected_records, (
    f"Record count mismatch: "
    f"Expected {expected_records}, found {actual_records}"
)

# Verify expected currency pairs
EXPECTED_CURRENCIES = {"USD", "EUR", "GBP"}

assert set(df_fx_raw["base_currency"]) == EXPECTED_CURRENCIES, (
    "Unexpected or missing base currencies"
)

assert df_fx_raw["quote_currency"].eq("INR").all(), (
    "Unexpected quote currency"
)

print("FX Data Loading: SUCCESS")
print("Metadata Loading: SUCCESS")
print("Total Records:", actual_records)
print("Expected Records:", expected_records)
print("Currency Pairs:", sorted(EXPECTED_CURRENCIES))
print("Record Count Reconciliation: PASSED")

print("\nData Types:")
print(df_fx_raw.dtypes)

print("\nFirst Five Records:")
display(df_fx_raw.head())


FX Data Loading: SUCCESS
Metadata Loading: SUCCESS
Total Records: 312
Expected Records: 312
Currency Pairs: ['EUR', 'GBP', 'USD']
Record Count Reconciliation: PASSED

Data Types:
observation_date    datetime64[us]
base_currency                  str
quote_currency                 str
exchange_rate              float64
source                         str
dtype: object

First Five Records:


,observation_date,base_currency,quote_currency,exchange_rate,source
0,2026-05-21,EUR,INR,111.5885,Frankfurter
1,2026-05-22,EUR,INR,110.9585,Frankfurter
2,2026-05-25,EUR,INR,110.8870,Frankfurter
3,2026-05-26,EUR,INR,111.3200,Frankfurter
4,2026-05-27,EUR,INR,111.3650,Frankfurter



## 2. Schema and Data Type Validation

### Objective
Verify that the FX dataset follows the required structure and contains appropriate data types for financial analysis.

### Expected Schema

| Column | Expected Data Type | Business Purpose |
|---|---|---|
| observation_date | datetime | Historical exchange-rate date |
| base_currency | string | Foreign currency being measured |
| quote_currency | string | INR reporting currency |
| exchange_rate | numeric | INR required for one unit of foreign currency |
| source | string | Data lineage and auditability |

### Validation Rules
1. The dataset must contain exactly five expected columns.
2. Observation dates must use a datetime data type.
3. Exchange rates must use a numeric data type.
4. Currency identifiers and source must use string-compatible data types.
5. Unexpected columns must be reported.


In [2]:

from pandas.api.types import (
    is_datetime64_any_dtype,
    is_numeric_dtype,
    is_string_dtype
)

# Define expected columns
EXPECTED_COLUMNS = [
    "observation_date",
    "base_currency",
    "quote_currency",
    "exchange_rate",
    "source"
]

# Build schema validation checks
schema_checks = {
    "Expected column count": len(df_fx_raw.columns) == 5,

    "Required columns present": (
        set(EXPECTED_COLUMNS).issubset(df_fx_raw.columns)
    ),

    "No unexpected columns": (
        set(df_fx_raw.columns) == set(EXPECTED_COLUMNS)
    ),

    "Observation date is datetime": (
        is_datetime64_any_dtype(
            df_fx_raw["observation_date"]
        )
    ),

    "Exchange rate is numeric": (
        is_numeric_dtype(
            df_fx_raw["exchange_rate"]
        )
    ),

    "Base currency is string": (
        is_string_dtype(
            df_fx_raw["base_currency"]
        )
    ),

    "Quote currency is string": (
        is_string_dtype(
            df_fx_raw["quote_currency"]
        )
    ),

    "Source is string": (
        is_string_dtype(
            df_fx_raw["source"]
        )
    )
}

# Convert results into a validation report
df_schema_validation = pd.DataFrame([
    {
        "validation_rule": rule,
        "status": "PASS" if passed else "FAIL"
    }
    for rule, passed in schema_checks.items()
])

# Display validation results
display(df_schema_validation)

# Summarize validation
passed_checks = (
    df_schema_validation["status"] == "PASS"
).sum()

total_checks = len(df_schema_validation)

print(f"Schema Checks Passed: {passed_checks}/{total_checks}")

# Stop if schema validation fails
assert all(schema_checks.values()), (
    "FX schema validation failed. Review the report."
)

print("FX Schema Validation: PASSED")


,validation_rule,status
0,Expected column count,PASS
1,Required columns present,PASS
2,No unexpected columns,PASS
3,Observation date is datetime,PASS
4,Exchange rate is numeric,PASS
5,Base currency is string,PASS
6,Quote currency is string,PASS
7,Source is string,PASS


Schema Checks Passed: 8/8
FX Schema Validation: PASSED



## 3. Missing Value Validation

### Business Objective
Identify missing values that could affect currency risk calculations, import-payment exposure analysis, and financial reporting.

### Method
1. Calculate missing values for each column.
2. Calculate the percentage of missing values.
3. Identify records containing at least one missing value.
4. Summarize the overall completeness of the dataset.

### Validation Rules
- observation_date must not be missing.
- base_currency must not be missing.
- quote_currency must not be missing.
- exchange_rate must not be missing.
- source must not be missing.

### Business Interpretation
Missing exchange rates may prevent accurate INR conversion and risk calculations.

Missing observation dates may prevent reliable time-series analysis.

Missing currency identifiers may cause records to be assigned to incorrect currency pairs.


In [3]:

# Calculate missing values by column
missing_counts = df_fx_raw.isna().sum()

# Calculate missing percentages
missing_percentages = (
    df_fx_raw.isna().mean() * 100
)

# Build missing-value validation report
df_missing_validation = pd.DataFrame({
    "column_name": df_fx_raw.columns,
    "missing_count": missing_counts.values,
    "missing_percentage": missing_percentages.values
})

# Add validation status
df_missing_validation["status"] = (
    df_missing_validation["missing_count"]
    .apply(lambda count: "PASS" if count == 0 else "FAIL")
)

# Identify rows containing missing values
rows_with_missing = df_fx_raw.isna().any(axis=1).sum()

# Calculate dataset completeness
total_cells = df_fx_raw.size
missing_cells = df_fx_raw.isna().sum().sum()

completeness_percentage = (
    (total_cells - missing_cells) / total_cells * 100
)

# Display report
print("FX Missing Value Validation Report:")
display(df_missing_validation)

print("\nTotal Records:", len(df_fx_raw))
print("Records With Missing Values:", rows_with_missing)
print("Total Missing Cells:", missing_cells)
print(f"Dataset Completeness: {completeness_percentage:.2f}%")

# Overall validation result
if missing_cells == 0:
    print("FX Missing Value Validation: PASSED")
else:
    print("FX Missing Value Validation: FAILED")
    print("Review missing records before processing.")


FX Missing Value Validation Report:


,column_name,missing_count,missing_percentage,status
0,observation_date,0,0.0,PASS
1,base_currency,0,0.0,PASS
2,quote_currency,0,0.0,PASS
3,exchange_rate,0,0.0,PASS
4,source,0,0.0,PASS



Total Records: 312
Records With Missing Values: 0
Total Missing Cells: 0
Dataset Completeness: 100.00%
FX Missing Value Validation: PASSED



## 4. Duplicate Record Validation

### Business Objective
Ensure that each currency pair has only one exchange-rate observation per date.

### Method
1. Identify exact duplicate rows across all columns.
2. Identify duplicate records using the business key:
   - observation_date
   - base_currency
   - quote_currency
3. Count duplicate records and affected business keys.
4. Report the validation results.

### Business Rules
- Exact duplicate rows are not permitted.
- Each currency pair must have exactly one record per available observation date.
- Duplicate currency-date combinations must be investigated before database loading.

### Business Impact
Duplicate exchange rates can inflate record counts, distort time-series calculations, and produce misleading financial reports.

In [4]:

# Define the business key
FX_BUSINESS_KEY = [
    "observation_date",
    "base_currency",
    "quote_currency"
]

# Check exact duplicate rows
exact_duplicate_count = int(
    df_fx_raw.duplicated().sum()
)

# Check duplicate business keys
business_key_duplicate_count = int(
    df_fx_raw.duplicated(
        subset=FX_BUSINESS_KEY
    ).sum()
)

# Identify all rows involved in duplicate business keys
duplicate_key_rows = df_fx_raw[
    df_fx_raw.duplicated(
        subset=FX_BUSINESS_KEY,
        keep=False
    )
].copy()

# Build validation report
df_duplicate_validation = pd.DataFrame([
    {
        "validation_rule": "Exact duplicate rows",
        "duplicate_count": exact_duplicate_count,
        "status": "PASS" if exact_duplicate_count == 0 else "FAIL"
    },
    {
        "validation_rule": "Duplicate currency-date business keys",
        "duplicate_count": business_key_duplicate_count,
        "status": "PASS" if business_key_duplicate_count == 0 else "FAIL"
    }
])

print("FX Duplicate Validation Report:")
display(df_duplicate_validation)

print("\nTotal Records:", len(df_fx_raw))
print("Exact Duplicate Rows:", exact_duplicate_count)
print(
    "Duplicate Business Keys:",
    business_key_duplicate_count
)
print(
    "Rows Involved in Duplicate Business Keys:",
    len(duplicate_key_rows)
)

# Display problematic records if found
if not duplicate_key_rows.empty:
    print("\nDuplicate Records Requiring Investigation:")
    display(
        duplicate_key_rows.sort_values(FX_BUSINESS_KEY)
    )

# Overall validation result
if exact_duplicate_count == 0 and business_key_duplicate_count == 0:
    print("\nFX Duplicate Validation: PASSED")
else:
    print("\nFX Duplicate Validation: FAILED")
    print("Investigate duplicate records before processing.")


FX Duplicate Validation Report:


,validation_rule,duplicate_count,status
0,Exact duplicate rows,0,PASS
1,Duplicate currency-date business keys,0,PASS



Total Records: 312
Exact Duplicate Rows: 0
Duplicate Business Keys: 0
Rows Involved in Duplicate Business Keys: 0

FX Duplicate Validation: PASSED



## 5. Exchange Rate Value Validation

### Business Objective
Verify that historical FX exchange rates are valid numerical values suitable for Treasury risk analysis.

### Method
1. Count zero exchange rates.
2. Count negative exchange rates.
3. Identify non-finite values such as infinity.
4. Check for non-numeric values.
5. Calculate minimum and maximum rates by currency pair.

### Validation Rules
- Exchange rates must be numeric.
- Exchange rates must be greater than zero.
- Exchange rates must be finite.
- Missing exchange rates are not permitted.

### Business Impact
Invalid exchange rates could produce incorrect INR import-payment costs, currency movements, and volatility calculations.

In [5]:

import numpy as np

# Convert rates to numeric for validation without modifying raw data
rates = pd.to_numeric(
    df_fx_raw["exchange_rate"],
    errors="coerce"
)

# Identify invalid values
non_numeric_count = int(
    (rates.isna() & df_fx_raw["exchange_rate"].notna()).sum()
)

missing_rate_count = int(rates.isna().sum())
zero_rate_count = int((rates == 0).sum())
negative_rate_count = int((rates < 0).sum())

non_finite_count = int(
    (rates.notna() & ~np.isfinite(rates)).sum()
)

# Build validation report
rate_checks = {
    "Non-numeric exchange rates": non_numeric_count,
    "Missing exchange rates": missing_rate_count,
    "Zero exchange rates": zero_rate_count,
    "Negative exchange rates": negative_rate_count,
    "Non-finite exchange rates": non_finite_count
}

df_rate_validation = pd.DataFrame([
    {
        "validation_rule": rule,
        "invalid_count": count,
        "status": "PASS" if count == 0 else "FAIL"
    }
    for rule, count in rate_checks.items()
])

print("FX Exchange Rate Validation Report:")
display(df_rate_validation)

# Summarize exchange-rate ranges by currency
df_rate_summary = (
    df_fx_raw.groupby("base_currency")["exchange_rate"]
    .agg(["count", "min", "max", "mean"])
    .round(4)
)

print("\nExchange Rate Summary by Currency:")
display(df_rate_summary)

# Overall validation result
if all(count == 0 for count in rate_checks.values()):
    print("\nFX Exchange Rate Value Validation: PASSED")
else:
    print("\nFX Exchange Rate Value Validation: FAILED")
    print("Investigate invalid exchange rates before processing.")


FX Exchange Rate Validation Report:


,validation_rule,invalid_count,status
0,Non-numeric exchange rates,0,PASS
1,Missing exchange rates,0,PASS
2,Zero exchange rates,0,PASS
3,Negative exchange rates,0,PASS
4,Non-finite exchange rates,0,PASS



Exchange Rate Summary by Currency:


,count,min,max,mean
base_currency,,,,
EUR,104,106.60,111.9595,109.7486
GBP,104,124.22,130.6900,127.9850
USD,104,94.33,96.7800,95.5121



FX Exchange Rate Value Validation: PASSED



## 6. Currency Pair and Data Source Validation

### Business Objective
Ensure that the FX dataset contains only the approved currencies used for international import payments.

### Validation Rules
1. Base currency must be USD, EUR, or GBP.
2. Quote currency must be INR.
3. Data source must be Frankfurter.
4. All three currency pairs must be represented.
5. Each currency pair must contain at least one observation.

### Business Impact
Incorrect currency identifiers could lead to inaccurate INR exposure calculations and misleading Treasury dashboards.

Incorrect source identifiers could compromise data lineage and auditability.

### Data Integrity
Any unexpected currencies or data sources must be investigated rather than automatically replaced.


In [6]:

# Define approved business values
APPROVED_BASE_CURRENCIES = {"USD", "EUR", "GBP"}
APPROVED_QUOTE_CURRENCY = "INR"
APPROVED_SOURCE = "Frankfurter"

# Validate currency and source fields
currency_checks = {
    "Approved base currencies": (
        df_fx_raw["base_currency"]
        .isin(APPROVED_BASE_CURRENCIES)
        .all()
    ),

    "INR quote currency": (
        df_fx_raw["quote_currency"]
        .eq(APPROVED_QUOTE_CURRENCY)
        .all()
    ),

    "Approved data source": (
        df_fx_raw["source"]
        .eq(APPROVED_SOURCE)
        .all()
    ),

    "All three currencies present": (
        set(df_fx_raw["base_currency"].unique())
        == APPROVED_BASE_CURRENCIES
    ),

    "Each currency has observations": (
        df_fx_raw.groupby("base_currency")
        .size()
        .gt(0)
        .all()
    )
}

# Create validation report
df_currency_validation = pd.DataFrame([
    {
        "validation_rule": rule,
        "status": "PASS" if passed else "FAIL"
    }
    for rule, passed in currency_checks.items()
])

print("Currency Pair and Source Validation:")
display(df_currency_validation)

print("\nRecords by Currency Pair:")
display(
    df_fx_raw.groupby(
        ["base_currency", "quote_currency"]
    ).size().reset_index(name="record_count")
)

# Overall validation result
passed_checks = sum(currency_checks.values())
total_checks = len(currency_checks)

print(f"\nChecks Passed: {passed_checks}/{total_checks}")

if all(currency_checks.values()):
    print("FX Currency and Source Validation: PASSED")
else:
    print("FX Currency and Source Validation: FAILED")
    print("Investigate unexpected currencies or sources.")


Currency Pair and Source Validation:


,validation_rule,status
0,Approved base currencies,PASS
1,INR quote currency,PASS
2,Approved data source,PASS
3,All three currencies present,PASS
4,Each currency has observations,PASS



Records by Currency Pair:


,base_currency,quote_currency,record_count
0,EUR,INR,104
1,GBP,INR,104
2,USD,INR,104



Checks Passed: 5/5
FX Currency and Source Validation: PASSED



## 7. FX Date Range and Coverage Validation

### Business Objective
Ensure that historical FX observations cover the requested analysis period and are consistently aligned across USD/INR, EUR/INR, and GBP/INR.

### Validation Method
1. Verify that observation dates fall within the requested period.
2. Identify the earliest and latest observation dates.
3. Calculate the number of unique dates per currency.
4. Compare observation-date sets across the three currencies.
5. Identify missing business-day observations for investigation.
6. Detect weekend observations.

### Validation Rules
- No observation may fall outside the requested date range.
- All three currency pairs must have identical observation-date sets.
- Date coverage must be documented.
- Missing business days must be investigated, not automatically filled.

### Business Impact
Misaligned currency dates can distort comparisons, daily percentage changes, volatility analysis, and Treasury risk reporting.

### Important Note
Weekends and financial-market holidays may not have published FX observations. Missing business-day dates are flagged for investigation rather than automatically classified as data-quality failures.


In [7]:

# Define the requested analysis period
START_DATE = pd.Timestamp("2026-05-21")
END_DATE = pd.Timestamp("2026-10-13")

# Check observations outside the requested date range
outside_range = df_fx_raw[
    ~df_fx_raw["observation_date"].between(
        START_DATE, END_DATE
    )
]

# Summarize coverage by currency
df_date_coverage = (
    df_fx_raw.groupby("base_currency")
    .agg(
        first_date=("observation_date", "min"),
        last_date=("observation_date", "max"),
        observation_count=("observation_date", "count"),
        unique_dates=("observation_date", "nunique")
    )
    .reset_index()
)

print("FX Date Coverage by Currency:")
display(df_date_coverage)

# Build observation-date sets
currency_date_sets = {
    currency: set(group["observation_date"])
    for currency, group in df_fx_raw.groupby("base_currency")
}

# Compare date sets across all currencies
date_sets_aligned = (
    len({
        frozenset(dates)
        for dates in currency_date_sets.values()
    }) == 1
    and set(currency_date_sets) == {"USD", "EUR", "GBP"}
)

# Identify dates present in some currencies but not others
all_observed_dates = set().union(
    *currency_date_sets.values()
)

missing_relative_dates = {
    currency: sorted(all_observed_dates - dates)
    for currency, dates in currency_date_sets.items()
}

# Generate Monday-Friday dates
expected_business_days = pd.bdate_range(
    start=START_DATE,
    end=END_DATE
)

# Identify missing weekdays per currency
missing_business_days = {
    currency: sorted(
        set(expected_business_days) - dates
    )
    for currency, dates in currency_date_sets.items()
}

# Identify weekend observations
weekend_records = df_fx_raw[
    df_fx_raw["observation_date"].dt.dayofweek >= 5
]

# Build validation report
date_checks = {
    "No dates outside requested period": len(outside_range) == 0,
    "Currency observation dates aligned": date_sets_aligned,
    "No weekend observations": len(weekend_records) == 0
}

df_date_validation = pd.DataFrame([
    {
        "validation_rule": rule,
        "status": "PASS" if passed else "FAIL"
    }
    for rule, passed in date_checks.items()
])

print("\nFX Date Validation Report:")
display(df_date_validation)

print("\nMissing Dates Relative to Other Currencies:")
for currency, dates in missing_relative_dates.items():
    print(f"{currency}: {len(dates)}")

print("\nMissing Monday-Friday Observations:")
for currency, dates in missing_business_days.items():
    print(
        f"{currency}: {len(dates)} | "
        f"{[d.strftime('%Y-%m-%d') for d in dates]}"
    )

print("\nWeekend Records:", len(weekend_records))
print("Dates Outside Requested Range:", len(outside_range))

if all(date_checks.values()):
    print("\nFX Date Range and Alignment Validation: PASSED")
else:
    print("\nFX Date Range and Alignment Validation: FAILED")
    print("Investigate the reported date issues.")


FX Date Coverage by Currency:


,base_currency,first_date,last_date,observation_count,unique_dates
0,EUR,2026-05-21,2026-10-13,104,104
1,GBP,2026-05-21,2026-10-13,104,104
2,USD,2026-05-21,2026-10-13,104,104



FX Date Validation Report:


,validation_rule,status
0,No dates outside requested period,PASS
1,Currency observation dates aligned,PASS
2,No weekend observations,PASS



Missing Dates Relative to Other Currencies:
EUR: 0
GBP: 0
USD: 0

Missing Monday-Friday Observations:
EUR: 0 | []
GBP: 0 | []
USD: 0 | []

Weekend Records: 0
Dates Outside Requested Range: 0

FX Date Range and Alignment Validation: PASSED



## 8. FX Statistical Outlier Screening

### Business Objective
Identify unusual exchange-rate observations that may affect currency-risk monitoring, volatility analysis, and import-payment planning.

### Method
Use the Interquartile Range (IQR) method independently for USD/INR, EUR/INR, and GBP/INR.

For each currency:
1. Calculate the first quartile (Q1).
2. Calculate the third quartile (Q3).
3. Calculate IQR = Q3 - Q1.
4. Calculate lower and upper outlier boundaries.
5. Flag exchange rates outside these boundaries.
6. Summarize the number of flagged observations.

### Validation Rules
- Outliers must be identified and documented.
- Flagged observations must not be automatically deleted.
- Genuine market movements must be preserved.
- Outlier counts are monitoring indicators, not automatic validation failures.

### Business Interpretation
Unusual exchange-rate levels may reflect genuine currency appreciation or depreciation.

Outlier screening helps identify observations requiring further investigation before financial reporting.

In [8]:

# Create a separate working copy
df_fx_outliers = df_fx_raw.copy()

# Calculate IQR boundaries independently for each currency
outlier_summary = []

df_fx_outliers["is_outlier"] = False

for currency, group in df_fx_outliers.groupby("base_currency"):

    q1 = group["exchange_rate"].quantile(0.25)
    q3 = group["exchange_rate"].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Flag observations outside the IQR boundaries
    outlier_mask = (
        (group["exchange_rate"] < lower_bound) |
        (group["exchange_rate"] > upper_bound)
    )

    df_fx_outliers.loc[
        group.index, "is_outlier"
    ] = outlier_mask

    outlier_summary.append({
        "currency": currency,
        "total_records": len(group),
        "q1": round(q1, 4),
        "q3": round(q3, 4),
        "iqr": round(iqr, 4),
        "lower_bound": round(lower_bound, 4),
        "upper_bound": round(upper_bound, 4),
        "outlier_count": int(outlier_mask.sum()),
        "outlier_percentage": round(
            outlier_mask.mean() * 100, 2
        )
    })

# Create summary report
df_outlier_summary = pd.DataFrame(outlier_summary)

print("FX Statistical Outlier Summary:")
display(df_outlier_summary)

# Extract flagged observations
df_flagged_outliers = df_fx_outliers[
    df_fx_outliers["is_outlier"]
].copy()

print("\nTotal FX Records:", len(df_fx_outliers))
print("Total Flagged Outliers:", len(df_flagged_outliers))

if not df_flagged_outliers.empty:
    print("\nFlagged Exchange Rate Observations:")
    display(
        df_flagged_outliers.sort_values(
            ["base_currency", "observation_date"]
        )
    )
else:
    print("\nNo statistical outliers detected.")

print("\nFX Statistical Outlier Screening: COMPLETED")


FX Statistical Outlier Summary:


,currency,total_records,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,EUR,104,108.974,110.5460,1.5720,106.6160,112.9040,1,0.96
1,GBP,104,127.325,128.9200,1.5950,124.9325,131.3125,5,4.81
2,USD,104,95.200,95.8375,0.6375,94.2437,96.7938,0,0.00



Total FX Records: 312
Total Flagged Outliers: 6

Flagged Exchange Rate Observations:


,observation_date,base_currency,quote_currency,exchange_rate,source,is_outlier
103,2026-10-13,EUR,INR,106.60,Frankfurter,True
124,2026-06-18,GBP,INR,124.80,Frankfurter,True
125,2026-06-19,GBP,INR,124.83,Frankfurter,True
128,2026-06-24,GBP,INR,124.60,Frankfurter,True
129,2026-06-25,GBP,INR,124.22,Frankfurter,True
130,2026-06-26,GBP,INR,124.78,Frankfurter,True



FX Statistical Outlier Screening: COMPLETED



## 9. FX Validation Summary and Processed Dataset Preparation

### Business Objective
Prepare a validated FX dataset for PostgreSQL loading, financial calculations, and Power BI reporting.

### Validation Findings
- Total FX observations: 312
- Schema validation: PASSED
- Missing-value validation: PASSED
- Duplicate validation: PASSED
- Exchange-rate value validation: PASSED
- Currency and source validation: PASSED
- Date-range and alignment validation: PASSED
- Statistical outliers flagged: 6

### Processing Decisions
1. Retain all 312 original observations.
2. Preserve the original exchange rates without modification.
3. Add a statistical outlier indicator.
4. Save a processed CSV separately from the raw extract.
5. Save validation summary information for auditability.

### Business Interpretation
The dataset passed the completed structural and business-rule checks. Six observations require further investigation because they were flagged by statistical screening.

The processed dataset is suitable for exploratory analysis with these flags retained. Statistical outliers must not be described as confirmed data errors.

### Expected Deliverables
- Data/processed/fx_rates_processed.csv
- Data/validation/fx_validation_summary.json
- Data/validation/fx_outlier_review.csv


In [9]:

from datetime import datetime, timezone

# Define output folders
PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"
VALIDATION_DIR = PROJECT_ROOT / "Data" / "validation"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# Prepare processed dataset without modifying raw data
df_fx_processed = df_fx_outliers.copy()

df_fx_processed = df_fx_processed.sort_values(
    ["base_currency", "observation_date"]
).reset_index(drop=True)

# Final record integrity checks
assert len(df_fx_processed) == len(df_fx_raw)
assert df_fx_processed["is_outlier"].sum() == 6
assert not df_fx_processed.duplicated(
    subset=FX_BUSINESS_KEY
).any()

# Save processed dataset
processed_path = PROCESSED_DIR / "fx_rates_processed.csv"

df_fx_processed.to_csv(
    processed_path,
    index=False
)

# Save flagged observations for further investigation
outlier_review_path = (
    VALIDATION_DIR / "fx_outlier_review.csv"
)

df_fx_processed[
    df_fx_processed["is_outlier"]
].to_csv(
    outlier_review_path,
    index=False
)

# Build validation summary
validation_summary = {
    "dataset": "FX Exchange Rates",
    "validation_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "total_records": int(len(df_fx_processed)),
    "currency_pairs": ["USD/INR", "EUR/INR", "GBP/INR"],
    "schema_validation": "PASSED",
    "missing_value_validation": "PASSED",
    "duplicate_validation": "PASSED",
    "exchange_rate_value_validation": "PASSED",
    "currency_source_validation": "PASSED",
    "date_coverage_validation": "PASSED",
    "statistical_outlier_screening": "COMPLETED",
    "flagged_outliers": int(
        df_fx_processed["is_outlier"].sum()
    ),
    "outlier_review_status": "PENDING",
    "records_removed": 0,
    "final_record_count": int(len(df_fx_processed))
}

# Save validation summary
summary_path = (
    VALIDATION_DIR / "fx_validation_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(
        validation_summary,
        file,
        indent=4
    )

# Verify saved dataset
df_fx_verified = pd.read_csv(processed_path)

assert len(df_fx_verified) == 312
assert df_fx_verified["is_outlier"].sum() == 6

print("FX Processed Dataset: SAVED")
print("FX Validation Summary: SAVED")
print("FX Outlier Review: SAVED")

print("\nFinal Records:", len(df_fx_verified))
print(
    "Flagged Outliers:",
    int(df_fx_verified["is_outlier"].sum())
)
print("Records Removed: 0")

print("\nProcessed Dataset Preview:")
display(df_fx_verified.head())


FX Processed Dataset: SAVED
FX Validation Summary: SAVED
FX Outlier Review: SAVED

Final Records: 312
Flagged Outliers: 6
Records Removed: 0

Processed Dataset Preview:


,observation_date,base_currency,quote_currency,exchange_rate,source,is_outlier
0,2026-05-21,EUR,INR,111.5885,Frankfurter,False
1,2026-05-22,EUR,INR,110.9585,Frankfurter,False
2,2026-05-25,EUR,INR,110.8870,Frankfurter,False
3,2026-05-26,EUR,INR,111.3200,Frankfurter,False
4,2026-05-27,EUR,INR,111.3650,Frankfurter,False


In [2]:

import pandas as pd

# Read the existing processed FX dataset.
fx_processed_path = (
    project_root / "Data" / "processed" / "fx_rates_processed.csv"
)

fx_df = pd.read_csv(fx_processed_path)

# Calculate data quality metrics.
record_count = len(fx_df)
missing_values = int(fx_df.isna().sum().sum())

duplicate_count = int(
    fx_df.duplicated(
        subset=[
            "observation_date",
            "base_currency",
            "quote_currency",
        ]
    ).sum()
)

outlier_count = int(fx_df["is_outlier"].sum())

# Record results in the pipeline log.
logger.info("FX validation results verified")
logger.info("Processed FX records: %s", record_count)
logger.info("Missing values: %s", missing_values)
logger.info("Duplicate currency-date records: %s", duplicate_count)
logger.info("Flagged FX outliers: %s", outlier_count)

if missing_values == 0 and duplicate_count == 0:
    logger.info("FX completeness and uniqueness checks PASSED")
else:
    logger.error("FX completeness or uniqueness check FAILED")


2026-11-02 16:06:12,426 | INFO | FX_Data_Validation | FX validation results verified
2026-11-02 16:06:12,428 | INFO | FX_Data_Validation | Processed FX records: 312
2026-11-02 16:06:12,430 | INFO | FX_Data_Validation | Missing values: 0
2026-11-02 16:06:12,431 | INFO | FX_Data_Validation | Duplicate currency-date records: 0
2026-11-02 16:06:12,432 | INFO | FX_Data_Validation | Flagged FX outliers: 6
2026-11-02 16:06:12,433 | INFO | FX_Data_Validation | FX completeness and uniqueness checks PASSED
